In [1]:
# ── Jupyter Notebook matplotlib 한글 폰트 설정 (macOS) ──────────────────────
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"  # macOS 기본 한글 폰트
plt.rcParams["axes.unicode_minus"] = False  # 마이너스 기호 깨짐 방지

print(f"현재 폰트: {plt.rcParams['font.family']}")

현재 폰트: ['AppleGothic']


## 0. train_voice.py 모듈 import

이 노트북은 `train_voice.py`를 **복사하지 않고 모듈로 import**해서 씁니다.
이전 버전은 학습 스크립트 전체를 노트북 셀에 그대로 붙여넣어 두었는데,
그 결과 원본이 두 곳(스크립트/노트북)에 존재하게 되어 한쪽만 수정되고
다른 쪽은 갱신 안 되는 문제가 반복됐습니다 (구조 피처 8→10개 변경,
`_apply_risk_floor` 추가, holdout 분리 로직 등이 노트북엔 반영 안 된 채
방치돼 있었음). 아래 셀 하나로 항상 최신 로직을 그대로 가져와 씁니다.


In [2]:
# ── train_voice.py 모듈 import ───────────────────────────────────────────────
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../VoiceModel").resolve()))

import train_voice as tv

print(f"train_voice 모듈 로드 완료 | DATA_PATH={tv.DATA_PATH}")

train_voice 모듈 로드 완료 | DATA_PATH=../Data/CallData/metadata_clean.csv


## 1. 학습된 아티팩트 + 데이터 로드

`train_voice.py` 실행이 끝나 `voice_model_artifact.pkl` / `voice_vectorizer.pkl`이
있어야 합니다. 데이터는 `tv.load_data()`를 그대로 재사용합니다 —
이렇게 하면 학습 스크립트와 **완전히 동일한 기준**으로 train/val/test/holdout이
나뉘고, holdout(`synthetic_new_holdout`, `synthetic_fp_stress`)이 재분할 풀에서
빠지는 것도 자동으로 보장됩니다 (예전 검증 셀처럼 `_leak_free_split`을 전체
df에 직접 돌리면 holdout이 다시 섞여 들어가는 실수가 났었음).


In [3]:
# ── 아티팩트 로드 ─────────────────────────────────────────────────────────────
model, vectorizer, threshold, classes, vectorizer_type = tv.load_artifacts()
phishing_idx = classes.index("phishing")

print(f"모델 로드 완료 | threshold={threshold} | vectorizer={vectorizer_type}")

# ── 데이터 로드 (train_voice.py와 동일 기준으로 4분할) ───────────────────────
df_train, df_val, df_test, df_holdout = tv.load_data(tv.DATA_PATH)

모델 로드 완료 | threshold=0.35 | vectorizer=word_ngram
[Load] 전체 3080건 (신규 holdout 665건 분리) | 재분할 풀 2415건
  Train   : 1799건 — phishing=538 (29.9%) | normal=1261 (70.1%)
  Val     : 327건  — phishing=115 (35.2%) | normal=212 (64.8%)
  Test    : 289건 — phishing=107 (37.0%) | normal=182 (63.0%)
  Holdout : 665건 — phishing=112 (16.8%) | normal=553 (83.2%)  (학습에 전혀 사용 안 됨)


## 2. 위험도 분포 시각화

`prob_phishing`(NB 원본 확률)과 `risk_score`(구조 피처 보정 반영 후 최종 점수)를
**둘 다** 계산합니다. 이전 버전은 보정 전 `prob_phishing`만 쓰고 있어서, 실제
서비스에서 쓰이는 `risk_score`와 분포가 달랐습니다 (보정으로 끌어올려진
MEDIUM/HIGH 구간이 차트에 반영이 안 됐음).

또한 train까지 섞인 전체 데이터셋으로 그리면 "이미 학습에 쓰인 데이터라
당연히 잘 맞는" 착시가 생기므로, **test + holdout(학습에 전혀 안 쓰인 데이터)만
따로** 그립니다. 이미지 파일 저장은 생략하고 노트북에 바로 표시합니다.


In [4]:
# ── 채점 대상: 학습에 쓰이지 않은 test + holdout만 사용 ─────────────────────
import numpy as np
import pandas as pd

df_eval = pd.concat([df_test, df_holdout], ignore_index=True)

struct = tv._extract_struct_features(df_eval["text_clean"])
X = tv.build_feature_matrix(vectorizer, df_eval["text_clean"], struct, fit=False)

prob_phishing = model.predict_proba(X)[:, phishing_idx]
raw_scores = (prob_phishing * 100).astype(int)
risk_scores = np.array(
    [
        tv._apply_risk_floor(s, struct[i], df_eval["text_clean"].iloc[i])
        for i, s in enumerate(raw_scores)
    ]
)

df_eval["prob_phishing"] = prob_phishing
df_eval["risk_score"] = risk_scores
df_eval["risk_level"] = df_eval["risk_score"].apply(tv._map_risk_level)

print(
    f"[Score] test+holdout {len(df_eval)}건 채점 완료 | "
    f"label={df_eval['label'].value_counts().to_dict()}"
)

ValueError: X has 8010 features, but ComplementNB is expecting 8008 features as input.

In [ ]:
# ── 색상 팔레트 ───────────────────────────────────────────────────────────────
COLOR_NORMAL = "#4C9BE8"
COLOR_PHISHING = "#E8534C"
COLOR_LOW = "#52B788"
COLOR_MEDIUM = "#F4A261"
COLOR_HIGH = "#E63946"

MEDIUM_LOWER = tv.RISK_MEDIUM_THRESHOLD / 100
HIGH_LOWER = tv.RISK_HIGH_THRESHOLD / 100

In [ ]:
# ── Fig 1: risk_score 히스토그램 + label별 KDE 비교 ──────────────────────────
import matplotlib.ticker as mticker
from scipy.stats import gaussian_kde

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    "보이스피싱 위험도(risk_score) 분포 — test + holdout",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)

scores_all = df_eval["risk_score"].values / 100
scores_phishing = df_eval.loc[df_eval["label"] == "phishing", "risk_score"].values / 100
scores_normal = df_eval.loc[df_eval["label"] == "normal", "risk_score"].values / 100

# 좌: 전체 히스토그램 + 구간 배경
ax = axes[0]
ax.hist(
    scores_all, bins=30, color="#7B9EC1", edgecolor="white", linewidth=0.6, alpha=0.85
)
for x, label, color in [
    (MEDIUM_LOWER, "LOW | MEDIUM", COLOR_MEDIUM),
    (HIGH_LOWER, "MEDIUM | HIGH", COLOR_HIGH),
]:
    ax.axvline(x, color=color, linewidth=1.8, linestyle="--", label=label)
ax.axvspan(0, MEDIUM_LOWER, alpha=0.08, color=COLOR_LOW, label="LOW")
ax.axvspan(MEDIUM_LOWER, HIGH_LOWER, alpha=0.08, color=COLOR_MEDIUM, label="MEDIUM")
ax.axvspan(HIGH_LOWER, 1.0, alpha=0.08, color=COLOR_HIGH, label="HIGH")
ax.set_title("전체 risk_score 히스토그램", fontsize=12)
ax.set_xlabel("risk_score")
ax.set_ylabel("샘플 수")
ax.legend(fontsize=8, loc="upper center")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

# 우: label별 KDE 오버레이
ax2 = axes[1]
for scores, label, color in [
    (scores_normal, "normal", COLOR_NORMAL),
    (scores_phishing, "phishing", COLOR_PHISHING),
]:
    kde = gaussian_kde(scores, bw_method=0.15)
    x_grid = np.linspace(0, 1, 300)
    ax2.fill_between(x_grid, kde(x_grid), alpha=0.35, color=color)
    ax2.plot(x_grid, kde(x_grid), color=color, linewidth=2, label=label)
for x, color in [(MEDIUM_LOWER, COLOR_MEDIUM), (HIGH_LOWER, COLOR_HIGH)]:
    ax2.axvline(x, color=color, linewidth=1.5, linestyle="--")
ax2.set_title("label별 risk_score 분포 비교 (KDE)", fontsize=12)
ax2.set_xlabel("risk_score")
ax2.set_ylabel("밀도 (density)")
ax2.legend(fontsize=10)
ax2.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

plt.tight_layout()
plt.savefig("risk_distribution_fig1.png", dpi=150, bbox_inches="tight")
plt.show()
print("[Save] risk_distribution_fig1.png")

In [ ]:
# ── Fig 2: 구간별 샘플 수 + label별 비율 ─────────────────────────────────────
ct = pd.crosstab(df_eval["risk_level"], df_eval["label"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    "위험도 구간(LOW / MEDIUM / HIGH) 분석 — test + holdout",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)

levels = ["LOW", "MEDIUM", "HIGH"]
x, width = np.arange(len(levels)), 0.35

ax = axes[0]
bars_n = ax.bar(
    x - width / 2,
    [ct.loc[lv, "normal"] if lv in ct.index else 0 for lv in levels],
    width,
    label="normal",
    color=COLOR_NORMAL,
    alpha=0.85,
    edgecolor="white",
)
bars_p = ax.bar(
    x + width / 2,
    [ct.loc[lv, "phishing"] if lv in ct.index else 0 for lv in levels],
    width,
    label="phishing",
    color=COLOR_PHISHING,
    alpha=0.85,
    edgecolor="white",
)
for bar in [*bars_n, *bars_p]:
    h = bar.get_height()
    if h > 0:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            h + 1,
            str(int(h)),
            ha="center",
            va="bottom",
            fontsize=9,
        )
for pos, color in zip(x, [COLOR_LOW, COLOR_MEDIUM, COLOR_HIGH]):
    ax.axvspan(pos - 0.5, pos + 0.5, alpha=0.06, color=color)
ax.set_title("구간별 샘플 수 (label 분리)", fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(["LOW\n(<40점)", "MEDIUM\n(40~69점)", "HIGH\n(≥70점)"], fontsize=10)
ax.set_ylabel("샘플 수")
ax.legend(fontsize=10)
ax.set_ylim(0, ax.get_ylim()[1] * 1.15)

ax2 = axes[1]
labels_order = ["normal", "phishing"]
x2 = np.arange(len(labels_order))
pct_by_label = (
    pd.crosstab(df_eval["label"], df_eval["risk_level"], normalize="index") * 100
)
for col in ["LOW", "MEDIUM", "HIGH"]:
    if col not in pct_by_label.columns:
        pct_by_label[col] = 0.0
bottom = np.zeros(len(labels_order))
for lv, color in [("LOW", COLOR_LOW), ("MEDIUM", COLOR_MEDIUM), ("HIGH", COLOR_HIGH)]:
    vals = [
        pct_by_label.loc[l, lv] if l in pct_by_label.index else 0 for l in labels_order
    ]
    bars = ax2.bar(
        x2, vals, bottom=bottom, label=lv, color=color, alpha=0.85, edgecolor="white"
    )
    for i, (bar, val) in enumerate(zip(bars, vals)):
        if val >= 3:
            ax2.text(
                bar.get_x() + bar.get_width() / 2,
                bottom[i] + val / 2,
                f"{val:.1f}%",
                ha="center",
                va="center",
                fontsize=9,
                fontweight="bold",
                color="white",
            )
    bottom += np.array(vals)
ax2.set_title("label별 risk_level 비율 (%)", fontsize=12)
ax2.set_xticks(x2)
ax2.set_xticklabels(labels_order, fontsize=11)
ax2.set_ylabel("비율 (%)")
ax2.set_ylim(0, 110)
ax2.legend(
    title="risk_level",
    fontsize=9,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.12),
    ncol=3,
)
ax2.yaxis.set_major_formatter(mticker.PercentFormatter())

plt.tight_layout()
plt.savefig("risk_distribution_fig2.png", dpi=150, bbox_inches="tight")
plt.show()
print("[Save] risk_distribution_fig2.png")

In [ ]:
# ── 콘솔 요약 ─────────────────────────────────────────────────────────────────
print("=" * 50)
print("[ 위험도 분포 요약 (test + holdout) ]")
print("=" * 50)
print(pd.crosstab(df_eval["risk_level"], df_eval["label"], margins=True))

print("\n[ 구간별 피싱 적중률 ]")
for lv in ["LOW", "MEDIUM", "HIGH"]:
    sub = df_eval[df_eval["risk_level"] == lv]
    if len(sub) == 0:
        continue
    phish_rate = (sub["label"] == "phishing").mean() * 100
    print(f"  {lv:6}: {len(sub):3}건 | 피싱 비율 {phish_rate:.1f}%")

## 3. 데이터 누수 재검증

`load_data()`가 내부적으로 수행하는 leak-free 분할이 실제로 깨끗한지
샘플 레벨에서 재확인합니다. `_leak_free_split`을 다시 호출하지 않고
**위에서 이미 로드한 `df_train`/`df_val`/`df_test`를 그대로 검사**합니다.


In [ ]:
# ── 누수 검증 A: parent_call_id 기준 ─────────────────────────────────────────
train_calls = set(df_train["parent_call_id"])
val_calls = set(df_val["parent_call_id"])
test_calls = set(df_test["parent_call_id"])

train_test_overlap = train_calls & test_calls
train_val_overlap = train_calls & val_calls
val_test_overlap = val_calls & test_calls

print("=== [누수 검증 A] parent_call_id 기준 통화 중복 ===\n")
print(f"Train ∩ Test  겹치는 통화 수: {len(train_test_overlap)}개")
print(f"Train ∩ Val   겹치는 통화 수: {len(train_val_overlap)}개")
print(f"Val   ∩ Test  겹치는 통화 수: {len(val_test_overlap)}개")

if train_test_overlap:
    leak_samples = df_test[df_test["parent_call_id"].isin(train_test_overlap)]
    print(
        f"\n[WARNING] Train-Test 누수 {len(train_test_overlap)}개 통화, "
        f"test에 누수된 샘플 {len(leak_samples)}건"
    )
    print(
        leak_samples[["sample_id", "parent_call_id", "label", "text_clean"]]
        .head(3)
        .to_string()
    )
else:
    print("\n[OK] Train-Test 간 통화 중복 없음")

In [ ]:
# ── 누수 검증 B: text_clean 기준 ─────────────────────────────────────────────
train_texts = set(df_train["text_clean"])
test_texts = set(df_test["text_clean"])
tt_text_overlap = train_texts & test_texts

print("=== [누수 검증 B] text_clean 기준 텍스트 중복 ===\n")
print(f"Train ∩ Test 겹치는 텍스트: {len(tt_text_overlap)}건")

if tt_text_overlap:
    print(f"\n[WARNING] Train-Test 텍스트 누수 {len(tt_text_overlap)}건")
    for t in list(tt_text_overlap)[:3]:
        print(f"  → {t[:80]}...")
else:
    print("[OK] Train-Test 간 텍스트 중복 없음")

## 4. 구조적 피처 기여도 확인

구조적 피처(계좌·긴급성 등 10개)를 0으로 채워 기여도를 제거했을 때
recall이 얼마나 달라지는지로, 모델이 키워드 패턴에만 과의존하는지 확인합니다.
`n_struct`를 하드코딩하지 않고 `tv.STRUCT_FEATURE_NAMES`에서 개수를 그대로
가져와서, 피처가 추가/삭제돼도 이 셀을 안 고쳐도 되게 했습니다.


In [ ]:
from scipy.sparse import csr_matrix, hstack
from sklearn.metrics import classification_report, confusion_matrix

N_STRUCT = len(tv.STRUCT_FEATURE_NAMES)  # 하드코딩 대신 train_voice.py 기준으로 도출


def build_X(texts: pd.Series, include_struct: bool = True):
    """텍스트 피처 + 구조적 피처 결합. include_struct=False면 구조적 피처를 0으로 채운다."""
    X_text = vectorizer.transform(texts)
    X_struct = (
        csr_matrix(tv._extract_struct_features(texts))
        if include_struct
        else csr_matrix(np.zeros((len(texts), N_STRUCT), dtype=np.float32))
    )
    return hstack([X_text, X_struct])


def evaluate_recall(X, y_true: pd.Series, label: str) -> dict:
    y_prob = model.predict_proba(X)[:, phishing_idx]
    y_pred = np.where(y_prob >= threshold, "phishing", "normal")
    cm = confusion_matrix(y_true, y_pred, labels=["normal", "phishing"])
    rec_p = cm[1, 1] / cm[1].sum() if cm[1].sum() > 0 else 0.0
    rec_n = cm[0, 0] / cm[0].sum() if cm[0].sum() > 0 else 0.0
    print(f"\n[{label}]")
    print(classification_report(y_true, y_pred, target_names=["normal", "phishing"]))
    return {"rec_p": rec_p, "rec_n": rec_n}


print("=== [성능 검증] 텍스트 단독 vs 전체 피처 비교 (test 세트) ===")

r_full = evaluate_recall(
    build_X(df_test["text_clean"], include_struct=True),
    df_test["label"],
    "전체 피처 (텍스트 + 구조적)",
)
r_text = evaluate_recall(
    build_X(df_test["text_clean"], include_struct=False),
    df_test["label"],
    "텍스트 피처만 (구조적 피처 제외)",
)

print("\n구조적 피처 기여도:")
print(f"  Recall(phishing) 차이: {r_full['rec_p'] - r_text['rec_p']:+.4f}")
print(f"  Recall(normal)   차이: {r_full['rec_n'] - r_text['rec_n']:+.4f}")
print("→ 차이가 크면 구조적 피처 의존도가 높음, 작으면 텍스트 n-gram 자체로 학습된 것")

## 5. 유형(call_type)별 성능

이전 버전은 `metadata_clean.csv`에 `call_type`이 없다고 가정하고
`source_dataset`에서 수동으로 재구성했는데, 지금은 `call_type` 컬럼이
이미 원본에 있으므로(대출사기형/수사기관 사칭형/택배사칭형 등) 그대로 씁니다.


In [ ]:
X_test_full = build_X(df_test["text_clean"], include_struct=True)
y_prob_test = model.predict_proba(X_test_full)[:, phishing_idx]

df_test_scored = df_test.copy()
df_test_scored["y_pred"] = np.where(y_prob_test >= threshold, "phishing", "normal")
df_test_scored["correct"] = df_test_scored["y_pred"] == df_test_scored["label"]

print("피싱 유형별 탐지율(Recall):")
phishing_test = df_test_scored[df_test_scored["label"] == "phishing"]
for call_type, grp in phishing_test.groupby("call_type"):
    print(f"  {call_type:15}: {grp['correct'].mean():.4f} ({len(grp)}건)")

print("\n정상 유형별 정확도:")
normal_test = df_test_scored[df_test_scored["label"] == "normal"]
for call_type, grp in normal_test.groupby("call_type"):
    print(f"  {call_type:15}: {grp['correct'].mean():.4f} ({len(grp)}건)")

## 6. 피처 중요도 (ComplementNB log-prob 차이)

ComplementNB는 SHAP `LinearExplainer`를 지원하지 않아서,
`feature_log_prob_` 클래스별 차이로 대체합니다
(`log P(feature|phishing) − log P(feature|normal)`, 값이 클수록 피싱 특징적).
구조 피처 이름은 `tv.STRUCT_FEATURE_NAMES`를 그대로 가져와 하드코딩 드리프트를 없앴습니다.


In [ ]:
feature_names = list(vectorizer.get_feature_names_out()) + tv.STRUCT_FEATURE_NAMES

# CalibratedClassifierCV → 내부 base_estimator 추출
base_nb = model.calibrated_classifiers_[0].estimator
normal_idx_nb = list(base_nb.classes_).index("normal")
phishing_idx_nb = list(base_nb.classes_).index("phishing")

log_prob_normal = base_nb.feature_log_prob_[normal_idx_nb]
log_prob_phishing = base_nb.feature_log_prob_[phishing_idx_nb]
phishing_score = log_prob_phishing - log_prob_normal

n_top = 20
top_phishing_idx = np.argsort(phishing_score)[::-1][:n_top]
top_normal_idx = np.argsort(phishing_score)[:n_top]

print("=== 피싱 판단 기여 상위 20 키워드 ===\n")
for rank, idx in enumerate(top_phishing_idx, 1):
    print(f"  {rank:2}. '{feature_names[idx]}'  (score={phishing_score[idx]:.4f})")

print("\n=== 정상 판단 기여 상위 20 키워드 ===\n")
for rank, idx in enumerate(top_normal_idx, 1):
    print(f"  {rank:2}. '{feature_names[idx]}'  (score={phishing_score[idx]:.4f})")

In [ ]:
# ── 시각화 (PNG로 저장 + 노트북에 표시) ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle(
    "ComplementNB 피처 중요도 (log P(phishing) − log P(normal))",
    fontsize=14,
    fontweight="bold",
)

p_names = [feature_names[i] for i in top_phishing_idx]
p_scores = [phishing_score[i] for i in top_phishing_idx]
axes[0].barh(
    range(n_top), p_scores[::-1], color="#E8534C", alpha=0.85, edgecolor="white"
)
axes[0].set_yticks(range(n_top))
axes[0].set_yticklabels(p_names[::-1], fontsize=9)
axes[0].set_title("피싱 기여 키워드 (높을수록 피싱 특징적)", fontsize=11)
axes[0].set_xlabel("log P(feature|phishing) − log P(feature|normal)")
axes[0].axvline(0, color="black", linewidth=0.8)

n_names = [feature_names[i] for i in top_normal_idx]
n_scores = [phishing_score[i] for i in top_normal_idx]
axes[1].barh(
    range(n_top), n_scores[::-1], color="#4C9BE8", alpha=0.85, edgecolor="white"
)
axes[1].set_yticks(range(n_top))
axes[1].set_yticklabels(n_names[::-1], fontsize=9)
axes[1].set_title("정상 기여 키워드 (낮을수록 정상 특징적)", fontsize=11)
axes[1].set_xlabel("log P(feature|phishing) − log P(feature|normal)")
axes[1].axvline(0, color="black", linewidth=0.8)

plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print("[Save] feature_importance.png")

## 7. 종합 판단

위 검증 결과를 모아 성능이 실제로 신뢰할 만한지 자동 요약합니다.


In [ ]:
print("=" * 60)
print("[ 종합 검증 결과 ]")
print("=" * 60)

issues = []

if train_test_overlap:
    issues.append(f"[누수 A] Train-Test 통화 중복 {len(train_test_overlap)}개")
if tt_text_overlap:
    issues.append(f"[누수 B] Train-Test 텍스트 중복 {len(tt_text_overlap)}건")

struct_contribution_p = r_full["rec_p"] - r_text["rec_p"]
struct_contribution_n = r_full["rec_n"] - r_text["rec_n"]
if abs(struct_contribution_p) > 0.1:
    issues.append(
        f"[피처 의존] 구조적 피처 제거 시 Recall(phishing) {struct_contribution_p:+.4f} 변화 "
        "— 키워드 패턴에 과의존 가능성"
    )

if issues:
    print("\n⚠️  발견된 문제:")
    for issue in issues:
        print(f"   {issue}")
    print("\n→ 성능이 과대평가됐을 가능성 있음. 위 문제 해결 후 재학습 권장.")
else:
    print("\n✅ 누수 없음 + 텍스트 n-gram 패턴 기반 학습 확인")
    print(
        f"   Recall(phishing)={r_full['rec_p']:.4f} | Recall(normal)={r_full['rec_n']:.4f}"
    )
    print("\n→ 성능이 실제 데이터 패턴을 학습한 결과로 판단됨.")
    print("   단, 실제 운영 환경에서 새로운 유형의 보이스피싱 등장 시 재학습 필요.")